In [1]:

import nest_asyncio
nest_asyncio.apply()  # 중첩 이벤트 루프 허용
import os
import json
import logging
import asyncio
import pandas as pd
import numpy as np
import openai
from datetime import datetime
import sys
from typing import List, Dict, Optional
# from tenacity import retry, stop_after_attempt, wait_exponential
import re
import gc
import matplotlib.pyplot as plt
from tqdm import tqdm
api = pd.read_csv('../../data/info.csv')
api_key = api.loc[1][1]

# 한글 폰트 설정
import matplotlib.font_manager as fm
import matplotlib as mpl

# 한글 폰트 경로 설정 (맥OS 기준)
font_path = '/System/Library/Fonts/AppleSDGothicNeo.ttc'  # 맥OS의 기본 한글 폰트
font_prop = fm.FontProperties(fname=font_path)

# matplotlib 기본 폰트 설정
plt.rc('font', family=font_prop.get_name())
mpl.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

# 폰트 확인
print(f"설정된 폰트: {font_prop.get_name()}")
print(f"사용 가능한 한글 폰트:")
for font in fm.findSystemFonts():
    if 'gothic' in font.lower() or 'gulim' in font.lower() or 'malgun' in font.lower() or 'batang' in font.lower():
        print(f" - {font}")

# pd.set_option('display.max_rows', 100)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_colwidth', 100)
# pd.set_option('display.max_colwidth', None)

from scipy.spatial.distance import cosine
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from scipy.special import softmax

df = pd.read_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', 
                 key='df')

/var/folders/7z/944bgjcs659fsp58nc954qgm0000gn/T/ipykernel_12630/289036805.py:19: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  api_key = api.loc[1][1]


설정된 폰트: Apple SD Gothic Neo
사용 가능한 한글 폰트:
 - /System/Library/Fonts/Supplemental/NotoSansGothic-Regular.ttf
 - /System/Library/Fonts/AppleSDGothicNeo.ttc
 - /System/Library/Fonts/Supplemental/AppleGothic.ttf


### 임베딩

#### 임베딩 텍스트 생성

In [2]:
def create_optimized_visit_text(row):
    """방문 데이터에서 임상적으로 중요한 정보를 추출하여 풍부한 맥락의 텍스트 생성"""
    
    # 안전하게 열 접근하는 헬퍼 함수
    def safe_access(row, key):
        if isinstance(row, dict):
            return row.get(key, None)
        else:
            try:
                return row[key] if key in row.index else None
            except:
                return None
    
    # 값 유효성 확인 헬퍼 함수
    def is_valid(value):
        return pd.notna(value) and value is not None and value != ''
    
    # 최종 임상 텍스트를 담을 섹션별 컨테이너
    clinical_sections = []
    
    # 1. 기본 정보 섹션
    patient_id = safe_access(row, '환자번호')
    visit_date = safe_access(row, '날짜')
    clinical_sections.append(f"환자번호: {patient_id}, 방문일: {visit_date}")
    
    # 2. 주호소 및 증상 섹션 (Core Symptoms)
    symptoms_context = []
    
    # 증상 위치와 종류 통합
    cc_location = safe_access(row, 'CC_location')
    cc_pain_type = safe_access(row, 'CC_pain_type')
    
    if is_valid(cc_location) and is_valid(cc_pain_type):
        symptoms_context.append(f"주호소: 환자는 {cc_location}에 {cc_pain_type}을 호소합니다.")
    elif is_valid(cc_location):
        symptoms_context.append(f"주호소 위치: {cc_location}")
    elif is_valid(cc_pain_type):
        symptoms_context.append(f"통증 유형: {cc_pain_type}")
    
    # 턱 관련 통증과 불편감 세부 정보 추가
    cc_painUncomp_desc_jaw = safe_access(row, 'CC_painUncomp_desc_jaw')
    if is_valid(cc_painUncomp_desc_jaw):
        symptoms_context.append(f"턱 관련 통증 및 불편감: {cc_painUncomp_desc_jaw}")
    
    # 기능적 제한 정보 추가
    cc_disable_desc_jaw = safe_access(row, 'CC_disable_desc_jaw')
    if is_valid(cc_disable_desc_jaw):
        symptoms_context.append(f"턱 기능 제한: {cc_disable_desc_jaw}")
    
    # 근육과 관절 관련 증상 추가
    cc_muscle_joint_desc_stress = safe_access(row, 'CC_muscle_joint_desc_stress')
    if is_valid(cc_muscle_joint_desc_stress):
        symptoms_context.append(f"근육 및 관절 상태: {cc_muscle_joint_desc_stress}")
    
    # 통증 강도와 지속 기간 정보
    cc_severity = safe_access(row, 'CC_severity')
    if is_valid(cc_severity):
        symptoms_context.append(f"통증 강도(1-5): {cc_severity}")
    
    cc_vas = safe_access(row, 'CC_vas')
    if is_valid(cc_vas):
        symptoms_context.append(f"VAS 통증 점수: {cc_vas}")
    
    cc_duration = safe_access(row, 'CC_duration')
    if is_valid(cc_duration):
        symptoms_context.append(f"증상 지속 기간: {cc_duration}")
    
    # 증상 섹션을 통합하여 전체 맥락에 추가
    if symptoms_context:
        clinical_sections.append("【증상 정보】\n" + "\n".join(symptoms_context))
    
    # 3. 병력 및 습관 섹션 (History & Habits)
    history_context = []
    
    # 치과 관련 과거력
    cc_dentalHistory_desc = safe_access(row, 'CC_dentalHistory_desc')
    if is_valid(cc_dentalHistory_desc):
        history_context.append(f"치과 병력: {cc_dentalHistory_desc}")
    
    # 클리닉 방문 이력
    cc_clinic_history_desc = safe_access(row, 'CC_clinic_history_desc')
    if is_valid(cc_clinic_history_desc):
        history_context.append(f"턱관절 관련 과거 치료: {cc_clinic_history_desc}")
    
    # 생활 습관 요인
    cc_factor_habbit = safe_access(row, 'CC_factor_habbit')
    if is_valid(cc_factor_habbit):
        history_context.append(f"생활 습관 요인: {cc_factor_habbit}")
    
    # 습관 관련 상세 정보
    habit_details = []
    
    habit_type = safe_access(row, '습관_habit_type')
    if is_valid(habit_type):
        habit_details.append(f"습관 유형: {habit_type}")
    
    habit_frequency = safe_access(row, '습관_frequency')
    if is_valid(habit_frequency):
        habit_details.append(f"습관 빈도: {habit_frequency}")
    
    habit_awareness = safe_access(row, '습관_awareness')
    if is_valid(habit_awareness):
        habit_details.append(f"습관 인지 여부: {habit_awareness}")
    
    habit_improvement = safe_access(row, '습관_improvement')
    if is_valid(habit_improvement):
        habit_details.append(f"습관 개선 상태: {habit_improvement}")
    
    if habit_details:
        history_context.append("습관 상세정보: " + ", ".join(habit_details))
    
    # 병력 및 습관 섹션을 통합하여 전체 맥락에 추가
    if history_context:
        clinical_sections.append("【병력 및 습관】\n" + "\n".join(history_context))
    
    # 4. 치료 관련 섹션 (Treatment)
    treatment_context = []
    
    # 치료 계획
    cc_treat_plan = safe_access(row, 'CC_treat_plan')
    if is_valid(cc_treat_plan):
        treatment_context.append(f"치료 계획: {cc_treat_plan}")
    
    # 약물 관련 정보
    medication_details = []
    
    medication_type = safe_access(row, '약_medication_type')
    if is_valid(medication_type):
        medication_details.append(f"약물 유형: {medication_type}")
    
    medication_frequency = safe_access(row, '약_frequency')
    if is_valid(medication_frequency):
        medication_details.append(f"복용 빈도: {medication_frequency}")
    
    medication_duration = safe_access(row, '약_duration')
    if is_valid(medication_duration):
        medication_details.append(f"복용 기간: {medication_duration}")
    
    medication_compliance = safe_access(row, '약_compliance')
    if is_valid(medication_compliance):
        medication_details.append(f"복약 순응도: {medication_compliance}")
    
    if medication_details:
        treatment_context.append("약물 정보: " + ", ".join(medication_details))
    
    # 장치 관련 정보
    device_details = []
    
    device_type = safe_access(row, '장치_device_type')
    if is_valid(device_type):
        device_details.append(f"장치 유형: {device_type}")
    
    device_usage_pattern = safe_access(row, '장치_usage_pattern')
    if is_valid(device_usage_pattern):
        device_details.append(f"사용 패턴: {device_usage_pattern}")
    
    device_duration = safe_access(row, '장치_duration')
    if is_valid(device_duration):
        device_details.append(f"사용 기간: {device_duration}")
    
    device_compliance = safe_access(row, '장치_compliance')
    if is_valid(device_compliance):
        device_details.append(f"장치 순응도: {device_compliance}")
    
    if device_details:
        treatment_context.append("장치 정보: " + ", ".join(device_details))
    
    # 치료 섹션을 통합하여 전체 맥락에 추가
    if treatment_context:
        clinical_sections.append("【치료 정보】\n" + "\n".join(treatment_context))
    
    # 5. 임상 측정 데이터 섹션 (Clinical Measurements)
    measurements_context = []
    
    # 입 벌림 관련 측정값
    opening_measurements = []
    
    cmo_before = safe_access(row, 'CMO_before')
    if is_valid(cmo_before):
        opening_measurements.append(f"편안한 개구량(현재): {cmo_before}mm")
    
    cmo_after = safe_access(row, 'CMO_after')
    if is_valid(cmo_after):
        opening_measurements.append(f"편안한 개구량(이후): {cmo_after}mm")
    
    mmo_before = safe_access(row, 'MMO_before')
    if is_valid(mmo_before):
        opening_measurements.append(f"최대 개구량(현재): {mmo_before}mm")
    
    mmo_after = safe_access(row, 'MMO_after')
    if is_valid(mmo_after):
        opening_measurements.append(f"최대 개구량(이후): {mmo_after}mm")
    
    if opening_measurements:
        measurements_context.append("개구량 측정: " + ", ".join(opening_measurements))
    
    # MMO-CMO 차이
    dif_MMO_CMO = safe_access(row, 'dif_MMO_CMO')
    if is_valid(dif_MMO_CMO):
        measurements_context.append(f"MMO-CMO 차이: {dif_MMO_CMO}mm")
    
    # 턱 편위 관련 정보
    deviation_info = []
    
    deviation_pattern = safe_access(row, 'deviation_pattern_type')
    if is_valid(deviation_pattern):
        deviation_info.append(f"편위 패턴: {deviation_pattern}")
    
    deviation_direction = safe_access(row, 'deviation_direction')
    if is_valid(deviation_direction):
        deviation_info.append(f"편위 방향: {deviation_direction}")
    
    deviation_intensity = safe_access(row, 'deviation_intensity')
    if is_valid(deviation_intensity):
        deviation_info.append(f"편위 강도: {deviation_intensity}")
    
    if deviation_info:
        measurements_context.append("턱 편위: " + ", ".join(deviation_info))
    
    # 통증 관련 측정값
    pain_measurements = []
    
    cap_pain_intensity = safe_access(row, 'Cap.pal_Pain_Intensity')
    if is_valid(cap_pain_intensity):
        pain_measurements.append(f"촉진 시 통증 강도: {cap_pain_intensity}")
    
    m_pain_intensity = safe_access(row, 'M.pal_Pain_Intensity')
    if is_valid(m_pain_intensity):
        pain_measurements.append(f"운동 시 통증 강도: {m_pain_intensity}")
    
    if pain_measurements:
        measurements_context.append("통증 측정: " + ", ".join(pain_measurements))
    
    # 소리 관련 정보
    noise_info = []
    
    noise_code = safe_access(row, 'Noise_Code')
    if is_valid(noise_code) and noise_code != 'No-Noise':
        noise_info.append(f"소리 유형: {noise_code}")
        
        noise_intensity = safe_access(row, 'Noise_Intensity')
        if is_valid(noise_intensity):
            noise_info.append(f"소리 강도: {noise_intensity}")
    
    if noise_info:
        measurements_context.append("턱관절 소리: " + ", ".join(noise_info))
    
    # 압흔 정보
    ridging_info = []
    
    tongue_ridging = safe_access(row, 'Tongue_ridging_Intensity')
    if is_valid(tongue_ridging):
        ridging_info.append(f"혀 압흔 강도: {tongue_ridging}")
    
    mucosal_ridging = safe_access(row, 'Mucosal_ridging_Intensity')
    if is_valid(mucosal_ridging):
        ridging_info.append(f"점막 압흔 강도: {mucosal_ridging}")
    
    if ridging_info:
        measurements_context.append("압흔 상태: " + ", ".join(ridging_info))
    
    # 측방 이동량
    lateral_info = []
    
    rt_before = safe_access(row, 'Rt_before')
    if is_valid(rt_before):
        lateral_info.append(f"우측 측방 이동량: {rt_before}mm")
    
    lt_before = safe_access(row, 'Lt_before')
    if is_valid(lt_before):
        lateral_info.append(f"좌측 측방 이동량: {lt_before}mm")
    
    if lateral_info:
        measurements_context.append("측방 이동량: " + ", ".join(lateral_info))
    
    # 임상 측정값 섹션을 통합하여 전체 맥락에 추가
    if measurements_context:
        clinical_sections.append("【임상 측정 데이터】\n" + "\n".join(measurements_context))
    
    # 모든 섹션을 통합하여 하나의 맥락화된 텍스트 생성
    return "\n\n".join(clinical_sections)

#### 일일 임베딩 함수

In [3]:
def create_visit_level_embeddings(df, api_key, batch_size=20):
    """각 방문 데이터를 독립적으로 임베딩하는 함수 (배치 처리 기능 추가)"""
    
    # OpenAI 클라이언트 초기화
    from openai import OpenAI
    client = OpenAI(api_key=api_key)
    
    # 결과 저장을 위한 리스트
    visit_embeddings = []
    
    # 전체 행 수
    total_rows = len(df)
    print(f"총 {total_rows}개 방문 데이터에 대한 임베딩을 생성합니다...")
    
    # 배치 단위로 처리
    for i in tqdm(range(0, total_rows, batch_size), desc="배치 처리 중"):
        # 현재 배치의 끝 인덱스 계산
        end_idx = min(i + batch_size, total_rows)
        batch_df = df.iloc[i:end_idx]
        
        batch_embeddings = []
        
        # 각 환자-방문 조합에 대해 처리
        for idx, row in batch_df.iterrows():
            patient_id = row['환자번호']
            visit_date = row['날짜']
            
            # 현재 방문의 텍스트 데이터 추출
            try:
                visit_text = create_optimized_visit_text(row)
                
                # 임베딩 생성
                response = client.embeddings.create(
                    model="text-embedding-ada-002",
                    input=visit_text
                )
                embedding = response.data[0].embedding
                
                # 결과 저장
                batch_embeddings.append({
                    'patient_id': patient_id,
                    'visit_date': visit_date,
                    'embedding': embedding,
                    'original_index': idx
                })
                
            except Exception as e:
                print(f"환자 {patient_id}, 방문일 {visit_date}: 임베딩 생성 실패 - {e}")
        
        # 배치 임베딩 리스트에 추가
        visit_embeddings.extend(batch_embeddings)
        
        # 중간 결과 저장 (안전장치)
        batch_filename = f'visit_embeddings_batch_{i//batch_size+1}.json'
        save_embeddings_to_json(batch_embeddings, batch_filename)
        print(f"배치 {i//batch_size+1}: {len(batch_embeddings)}개 방문 데이터의 임베딩이 '{batch_filename}'에 저장되었습니다.")
    
    return visit_embeddings

#### 임베딩 JSON 저장 함수

In [4]:
def save_embeddings_to_json(embeddings, filename):
    """임베딩을 JSON 파일로 저장하는 함수"""
    import json
    import numpy as np
    from datetime import datetime
    
    # 저장 경로 설정 (기본 경로에 embeddings 폴더)
    import os
    save_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
    os.makedirs(save_dir, exist_ok=True)
    filepath = os.path.join(save_dir, filename)
    
    # JSON 직렬화를 위해 NumPy 배열과 Timestamp 객체 처리
    serializable_embeddings = []
    for item in embeddings:
        serializable_item = item.copy()
        
        # 임베딩 벡터가 NumPy 배열인 경우 리스트로 변환
        if 'embedding' in serializable_item and hasattr(serializable_item['embedding'], 'tolist'):
            serializable_item['embedding'] = serializable_item['embedding'].tolist()
        
        # Timestamp 객체가 있는 경우 문자열로 변환
        if 'visit_date' in serializable_item and hasattr(serializable_item['visit_date'], 'strftime'):
            serializable_item['visit_date'] = serializable_item['visit_date'].strftime('%Y-%m-%d')
        
        serializable_embeddings.append(serializable_item)
    
    # 커스텀 JSON 인코더 사용
    class CustomJSONEncoder(json.JSONEncoder):
        def default(self, obj):
            # pandas Timestamp 객체 처리
            if hasattr(obj, 'strftime'):
                return obj.strftime('%Y-%m-%d')
            # NumPy 배열 처리
            if hasattr(obj, 'tolist'):
                return obj.tolist()
            return super().default(obj)
    
    try:
        with open(filepath, 'w', encoding='utf-8') as f:
            json.dump(serializable_embeddings, f, cls=CustomJSONEncoder, ensure_ascii=False, indent=2)
        return True
    except Exception as e:
        print(f"임베딩 저장 중 오류 발생: {e}")
        # 오류 시 백업 저장 시도
        backup_filepath = os.path.join(save_dir, f"backup_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{filename}")
        try:
            with open(backup_filepath, 'w', encoding='utf-8') as f:
                json.dump(serializable_embeddings, f, cls=CustomJSONEncoder, ensure_ascii=False, indent=2)
            print(f"백업 파일에 저장 성공: {backup_filepath}")
            return True
        except:
            print("백업 저장도 실패했습니다.")
            return False

#### 임베딩 파일들 호출 함수

In [5]:
def load_visit_embeddings(embedding_files):
    """저장된 임베딩 파일들을 로드하는 함수"""
    import json
    import os
    import numpy as np
    
    # 결과 저장을 위한 리스트
    all_embeddings = []
    
    # 각 파일 처리
    for file_path in embedding_files:
        if not os.path.exists(file_path):
            print(f"경고: {file_path} 파일이 존재하지 않습니다.")
            continue
            
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            if isinstance(data, list):
                for item in data:
                    if 'embedding' in item:
                        # 문자열이나 리스트로 저장된 임베딩을 NumPy 배열로 변환
                        if isinstance(item['embedding'], list):
                            item['embedding'] = np.array(item['embedding'])
                
                print(f"{file_path}에서 {len(data)}개 임베딩 로드 완료")
                all_embeddings.extend(data)
            else:
                print(f"경고: {file_path}의 형식이 예상과 다릅니다.")
                
        except Exception as e:
            print(f"{file_path} 로드 중 오류 발생: {e}")
    
    print(f"총 {len(all_embeddings)}개 임베딩 로드 완료")
    return all_embeddings

### 군집화

#### 군집화 실행

In [6]:
def form_clusters_from_visit_embeddings(visit_embeddings, n_clusters=5, random_state=42):
    """방문 임베딩으로부터 군집 형성"""
    
    # 임베딩 벡터 추출
    embeddings = np.array([item['embedding'] for item in visit_embeddings])
    
    # K-means 군집화
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    clusters = kmeans.fit_predict(embeddings)
    
    # 각 방문에 군집 할당
    for i, item in enumerate(visit_embeddings):
        item['cluster'] = int(clusters[i])
    
    # 군집 중심 반환
    cluster_centers = kmeans.cluster_centers_
    
    return visit_embeddings, cluster_centers

#### 환자 & 방문 레코드들 군집 분배

In [7]:
def calculate_cluster_probabilities(visit_embeddings, cluster_centers, temperature=0.1):
    """각 방문의 모든 군집에 대한 소속 확률 계산"""
    
    # 결과 리스트
    visit_with_probs = []
    
    for visit in visit_embeddings:
        embedding = np.array(visit['embedding'])
        
        # 모든 군집 중심과의 코사인 유사도 계산
        similarities = []
        for center in cluster_centers:
            similarity = 1 - cosine(embedding, center)  # 코사인 거리를 유사도로 변환
            similarities.append(similarity)
        
        # 유사도를 확률로 변환 (온도 조정된 소프트맥스)
        probabilities = softmax(np.array(similarities) / temperature)
        
        # 결과 저장
        visit_result = visit.copy()
        visit_result['cluster_probabilities'] = {i: float(prob) for i, prob in enumerate(probabilities)}
        visit_with_probs.append(visit_result)
    
    return visit_with_probs

In [8]:
def add_cluster_probabilities_to_dataframe(df, visit_with_probs):
    """원본 데이터프레임에 군집 확률 열 추가"""
    
    # 군집 수 결정
    num_clusters = len(next(iter(visit_with_probs))['cluster_probabilities'])
    
    # 새로운 확률 열 초기화
    for i in range(num_clusters):
        df[f'cluster_{i}_prob'] = 0.0
    
    # 각 방문의 확률 할당
    for visit in visit_with_probs:
        original_idx = visit['original_index']
        for cluster_id, prob in visit['cluster_probabilities'].items():
            df.loc[original_idx, f'cluster_{cluster_id}_prob'] = prob
    
    return df

### 임베딩-군집화 실행 파이프라인

In [9]:
def run_visit_level_clustering_pipeline(df, api_key, n_clusters=5, temperature=0.1):
    """방문별 임베딩 및 군집화 파이프라인 실행 함수 (개선된 버전)"""
    import os
    
    # 임베딩 파일 경로 설정
    embedding_dir = '/Users/nam-yeong/git/prj_centum/gpt_word/final_result/embeddings'
    visit_embedding_pattern = 'visit_embeddings_batch_*.json'
    embedding_files = []
    
    # 기존 임베딩 파일 확인
    import glob
    existing_files = glob.glob(os.path.join(embedding_dir, visit_embedding_pattern))
    
    # 1. 방문별 임베딩 생성 또는 로드
    if existing_files:
        print(f"기존 방문 임베딩 파일 {len(existing_files)}개가 발견되었습니다.")
        print("임베딩 파일을 로드합니다...")
        visit_embeddings = load_visit_embeddings(existing_files)
    else:
        print("방문별 임베딩 생성 중...")
        visit_embeddings = create_visit_level_embeddings(df, api_key, batch_size=20)
        
        # 통합 파일 저장
        save_embeddings_to_json(
            visit_embeddings, 
            f'visit_embeddings_complete_{len(visit_embeddings)}visits.json'
        )
    
    # 임베딩이 없는 경우 처리
    if not visit_embeddings:
        raise ValueError("유효한 임베딩을 생성하거나 로드할 수 없습니다.")
    
    # 2. 군집 형성
    print(f"{n_clusters}개 군집 형성 중...")
    visit_embeddings, cluster_centers = form_clusters_from_visit_embeddings(
        visit_embeddings, n_clusters=n_clusters
    )
    
    # 3. 각 방문의 군집 소속 확률 계산
    print("군집 소속 확률 계산 중...")
    visit_with_probs = calculate_cluster_probabilities(
        visit_embeddings, cluster_centers, temperature=temperature
    )
    
    # 4. 원본 데이터프레임에 확률 추가
    print("데이터프레임에 군집 확률 열 추가 중...")
    df_with_probs = add_cluster_probabilities_to_dataframe(df, visit_with_probs)
    
    # 5. 군집 특성 분석
    print("군집 특성 분석 중...")
    cluster_features = analyze_cluster_characteristics(df_with_probs, n_clusters)
    
    print("방문별 군집화 및 확률 할당 완료!")
    return df_with_probs, cluster_centers, cluster_features

In [10]:
def analyze_cluster_characteristics(df_with_probs, n_clusters):
    """각 군집의 특성 분석"""
    cluster_features = {}
    
    # 각 군집에 대해
    for cluster_id in range(n_clusters):
        # 군집에 주로 속한 방문 (50% 이상 확률)
        cluster_visits = df_with_probs[df_with_probs[f'cluster_{cluster_id}_prob'] > 0.5]
        
        if len(cluster_visits) == 0:
            cluster_features[cluster_id] = {
                'size': 0,
                'description': "충분한 데이터 없음"
            }
            continue
        
        # 주요 통계 계산
        features = {
            'size': len(cluster_visits),
            'avg_vas': cluster_visits['CC_vas'].mean() if 'CC_vas' in cluster_visits.columns else None,
            'avg_cmo': cluster_visits['CMO_before'].mean() if 'CMO_before' in cluster_visits.columns else None,
            'avg_mmo': cluster_visits['MMO_before'].mean() if 'MMO_before' in cluster_visits.columns else None,
            'common_pain_types': cluster_visits['CC_pain_type'].value_counts(normalize=True).head(3).to_dict() if 'CC_pain_type' in cluster_visits.columns else None,
            'common_locations': cluster_visits['CC_location'].value_counts(normalize=True).head(3).to_dict() if 'CC_location' in cluster_visits.columns else None,
        }
        
        # 군집 설명 생성
        description = generate_cluster_description(features)
        features['description'] = description
        
        cluster_features[cluster_id] = features
    
    return cluster_features

In [11]:
def generate_cluster_description(features):
    """군집 특성을 기반으로 설명 생성"""
    parts = []
    
    # VAS 기반 설명
    if features['avg_vas'] is not None:
        if features['avg_vas'] < 3:
            parts.append("낮은 통증")
        elif features['avg_vas'] < 6:
            parts.append("중간 정도 통증")
        else:
            parts.append("높은 통증")
    
    # CMO 기반 설명
    if features['avg_cmo'] is not None:
        if features['avg_cmo'] < 30:
            parts.append("제한된 개구량")
        elif features['avg_cmo'] < 40:
            parts.append("중간 개구량")
        else:
            parts.append("정상에 가까운 개구량")
    
    # 통증 유형 기반 설명
    if features['common_pain_types'] and len(features['common_pain_types']) > 0:
        top_pain = next(iter(features['common_pain_types']))
        parts.append(f"{top_pain} 유형 우세")
    
    # 통합 설명
    if parts:
        return ", ".join(parts)
    else:
        return "특성 정보 부족"

### 군집 및 패턴 식별

#### 완치 환자 식별

In [12]:
def identify_recovered_patients(df, recovery_threshold=70):
    """기존 calculate_composite_recovery_score 함수를 활용한 완치 환자 식별 함수
    
    Parameters:
    -----------
    df : DataFrame
        환자 데이터가 포함된 데이터프레임
    recovery_threshold : float, optional (default=70)
        완치로 간주할 복합 점수 임계값
    
    Returns:
    --------
    list
        완치된 환자 ID 목록
    dict
        환자별 완치 점수 및 등급 정보
    """
    # 환자별 마지막 방문 데이터 추출
    patient_last_visits = df.sort_values('날짜').groupby('환자번호').last().reset_index()
    
    # 결과 저장할 컨테이너
    recovered_patients = []
    patient_details = {}
    recovery_grades = {
        "완전 회복 (Complete Recovery)": 0,
        "상당한 회복 (Substantial Recovery)": 0,
        "중간 회복 (Moderate Recovery)": 0,
        "경미한 회복 (Mild Recovery)": 0,
        "최소 회복 (Minimal Recovery)": 0,
        "회복 미미 (Little to No Recovery)": 0
    }
    
    # 각 환자에 대해 복합 점수 계산
    for _, row in patient_last_visits.iterrows():
        patient_id = row['환자번호']
        
        # 기존 calculate_composite_recovery_score 함수 활용
        composite_score, details = calculate_composite_recovery_score(row)
        
        # 상세 정보 저장
        patient_details[patient_id] = {
            'composite_score': composite_score,
            'recovery_grade': details['recovery_grade'],
            'score_breakdown': details['scores']
        }
        
        # 회복 등급 통계 업데이트
        recovery_grades[details['recovery_grade']] += 1
        
        # 임계값 이상인 환자는 회복된 것으로 간주
        if composite_score >= recovery_threshold:
            recovered_patients.append(patient_id)
    
    # 결과 출력
    print(f"복합 완치 점수 {recovery_threshold} 이상인 환자: {len(recovered_patients)}명")
    print(f"회복 등급 분포:")
    for grade, count in recovery_grades.items():
        print(f"  - {grade}: {count}명")
    
    return recovered_patients, patient_details

#### 완치 환자 군집 확률 궤적 분석

In [13]:
def analyze_recovered_patient_trajectories(df_with_probs, recovered_ids, n_clusters=5):
    """완치 환자들의 군집 확률 궤적 분석"""
    # 완치 환자 데이터 필터링
    recovered_df = df_with_probs[df_with_probs['환자번호'].isin(recovered_ids)]
    
    # 치료 단계별 확률 패턴 분석
    stage_patterns = {}
    
    # 각 환자별로 치료 과정을 정규화하여 분석
    for patient_id, patient_visits in recovered_df.groupby('환자번호'):
        # 날짜순 정렬
        patient_visits = patient_visits.sort_values('날짜')
        
        # 치료 기간 계산
        treatment_days = (patient_visits['날짜'].max() - patient_visits['날짜'].min()).days
        if treatment_days == 0:  # 방문이 하루만 있는 경우
            continue
            
        # 방문별 상대적 진행 단계 계산 (0~100%)
        start_date = patient_visits['날짜'].min()
        
        for _, visit in patient_visits.iterrows():
            progress = ((visit['날짜'] - start_date).days / treatment_days * 100)
            stage = min(int(progress // 10) * 10, 90)  # 10% 단위로 구분 (0%, 10%, ..., 90%)
            
            if stage not in stage_patterns:
                stage_patterns[stage] = {f'cluster_{i}_prob': [] for i in range(n_clusters)}
            
            for i in range(n_clusters):
                prob_col = f'cluster_{i}_prob'
                if prob_col in visit:
                    stage_patterns[stage][prob_col].append(visit[prob_col])
    
    # 단계별 평균 확률 계산
    avg_stage_patterns = {}
    for stage, probs in stage_patterns.items():
        avg_stage_patterns[stage] = {
            cluster: np.mean(values) if values else 0 
            for cluster, values in probs.items()
        }
    
    # 데이터 시각화
    visualize_recovery_trajectories(avg_stage_patterns, n_clusters)
    
    return avg_stage_patterns

In [14]:
def visualize_recovery_trajectories(avg_stage_patterns, n_clusters):
    """완치 환자들의 치료 단계별 군집 확률 시각화"""
    import matplotlib.pyplot as plt
    
    # 데이터 준비
    stages = sorted(avg_stage_patterns.keys())
    cluster_probs = {f'cluster_{i}_prob': [] for i in range(n_clusters)}
    
    for stage in stages:
        for i in range(n_clusters):
            prob_col = f'cluster_{i}_prob'
            cluster_probs[prob_col].append(avg_stage_patterns[stage][prob_col])
    
    # 시각화
    plt.figure(figsize=(12, 6))
    for i in range(n_clusters):
        prob_col = f'cluster_{i}_prob'
        plt.plot(stages, cluster_probs[prob_col], marker='o', linewidth=2, label=f'군집 {i}')
    
    plt.title('완치 환자의 치료 단계별 군집 소속 확률 변화', fontsize=14)
    plt.xlabel('치료 진행 단계 (%)', fontsize=12)
    plt.ylabel('평균 군집 소속 확률', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()
    plt.xticks(stages)
    plt.tight_layout()
    plt.show()
    
    return plt

### 완치 궤적을 기반으로한 회복 예측 모델 구축

#### 많은 데이터 case : 로지스틱 회귀

In [15]:
def build_recovery_prediction_model(df_with_probs, recovered_ids, clinical_metrics=None):
    """완치 궤적 기반 회복 예측 모델 구축"""
    # 임상 지표가 제공되지 않은 경우 기본값 설정
    if clinical_metrics is None:
        clinical_metrics = ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']
    
    # 완치/비완치 환자 분류
    df_with_probs['is_recovered'] = df_with_probs['환자번호'].isin(recovered_ids)
    
    # 사용할 특성 선택
    cluster_cols = [col for col in df_with_probs.columns if col.startswith('cluster_') and col.endswith('_prob')]
    metrics_to_use = [metric for metric in clinical_metrics if metric in df_with_probs.columns]
    if not metrics_to_use:
        print("경고: 제공된 임상 지표가 데이터프레임에 없습니다. 군집 확률만 사용합니다.")
    
    feature_cols = cluster_cols + metrics_to_use
    
    # 환자 횟수가 적을 경우 로지스틱 회귀 대신 간단한 가중 모델 사용
    if len(df_with_probs['환자번호'].unique()) < 30:
        print("환자 수가 적어 통계적 회귀 모델 대신 가중치 기반 모델을 사용합니다.")
        return build_simple_weighted_model(df_with_probs, recovered_ids, feature_cols)
    
    # 충분한 데이터가 있으면 로지스틱 회귀 수행
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split, cross_val_score
    from sklearn.metrics import classification_report, roc_auc_score
    
    # 특성 및 타겟 데이터 준비
    X = df_with_probs[feature_cols].fillna(0)
    y = df_with_probs['is_recovered']
    
    # 환자별로 데이터 분할을 위한 그룹 정보
    groups = df_with_probs['환자번호'].values
    
    # 모델 초기화
    model = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000)
    
    # 교차 검증 (환자별 그룹화 고려)
    try:
        from sklearn.model_selection import GroupKFold
        gkf = GroupKFold(n_splits=5)
        cv_scores = cross_val_score(model, X, y, cv=gkf.split(X, y, groups), scoring='roc_auc')
        print(f"교차 검증 ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    except Exception as e:
        print(f"환자별 교차 검증 실패: {e}")
        # 일반 교차 검증으로 대체
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
        print(f"일반 교차 검증 ROC-AUC: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
    
    # 전체 데이터로 모델 학습
    model.fit(X, y)
    
    # 특성 중요도 계산
    feature_importance = {
        feature: importance for feature, importance in zip(feature_cols, model.coef_[0])
    }
    
    # 중요도 시각화
    visualize_feature_importance(feature_importance)
    
    return model, feature_importance

#### 적은 데이터 case : 가중 모델

In [16]:
def build_simple_weighted_model(df_with_probs, recovered_ids, feature_cols):
    """적은 데이터에 적합한 간단한 가중 모델 구축"""
    # 완치 환자와 비완치 환자 구분
    recovered_df = df_with_probs[df_with_probs['환자번호'].isin(recovered_ids)]
    non_recovered_df = df_with_probs[~df_with_probs['환자번호'].isin(recovered_ids)]
    
    # 각 특성의 평균값 계산
    recovered_means = recovered_df[feature_cols].mean()
    non_recovered_means = non_recovered_df[feature_cols].mean()
    
    # 특성별 차이 계산
    feature_diffs = recovered_means - non_recovered_means
    
    # 효과 크기 계산 (Standardized Mean Difference)
    recovered_std = recovered_df[feature_cols].std()
    non_recovered_std = non_recovered_df[feature_cols].std()
    
    # 0으로 나누기 방지
    pooled_std = np.sqrt((recovered_std**2 + non_recovered_std**2) / 2)
    pooled_std = pooled_std.replace(0, 1)  # 0인 경우 1로 대체
    
    effect_sizes = feature_diffs / pooled_std
    
    # 효과 크기로 가중치 정규화
    total_effect = np.abs(effect_sizes).sum()
    if total_effect == 0:
        weights = pd.Series(1/len(effect_sizes), index=effect_sizes.index)
    else:
        weights = np.abs(effect_sizes) / total_effect
    
    # 모델 함수 정의 (가중 평균 접근법)
    def predict_proba(X):
        # 정규화된 특성 생성
        X_norm = X.copy()
        for col in X.columns:
            mean_val = (recovered_means[col] + non_recovered_means[col]) / 2
            std_val = pooled_std[col]
            X_norm[col] = (X[col] - mean_val) / std_val
        
        # 가중 점수 계산
        scores = np.zeros(len(X))
        for col in X.columns:
            direction = np.sign(effect_sizes[col])
            scores += direction * X_norm[col] * weights[col]
        
        # 확률로 변환
        from scipy.special import expit  # 시그모이드 함수
        probs = expit(scores)
        
        # [not_recovered_prob, recovered_prob] 형태로 반환
        return np.column_stack([1 - probs, probs])
    
    # 간단한 모델 객체 생성
    class SimpleWeightedModel:
        def __init__(self, weights, effect_sizes, recovered_means, non_recovered_means, pooled_std, feature_names):
            self.weights = weights
            self.effect_sizes = effect_sizes
            self.recovered_means = recovered_means
            self.non_recovered_means = non_recovered_means
            self.pooled_std = pooled_std
            self.feature_names_in_ = feature_names
        
        def predict_proba(self, X):
            return predict_proba(X)
        
        def predict(self, X):
            probs = self.predict_proba(X)
            return (probs[:, 1] >= 0.5).astype(int)
    
    model = SimpleWeightedModel(
        weights, effect_sizes, recovered_means, non_recovered_means, 
        pooled_std, feature_cols
    )
    
    # 특성 중요도 (가중치로 대체)
    feature_importance = {
        feature: weight for feature, weight in zip(feature_cols, weights)
    }
    
    # 중요도 시각화
    visualize_feature_importance(feature_importance)
    
    return model, feature_importance

#### 특성 중요도 시각화

In [17]:
def visualize_feature_importance(feature_importance):
    """특성 중요도 시각화"""
    import matplotlib.pyplot as plt
    
    # 중요도 정렬
    sorted_importance = sorted(feature_importance.items(), key=lambda x: abs(x[1]), reverse=True)
    features = [x[0] for x in sorted_importance]
    importance = [x[1] for x in sorted_importance]
    
    # 너무 많은 특성이 있으면 상위 10개만 표시
    if len(features) > 10:
        features = features[:10]
        importance = importance[:10]
    
    # 시각화
    plt.figure(figsize=(10, 6))
    colors = ['g' if imp > 0 else 'r' for imp in importance]
    bars = plt.barh(range(len(features)), [abs(imp) for imp in importance], color=colors)
    
    plt.yticks(range(len(features)), features)
    plt.xlabel('특성 중요도 (절대값)', fontsize=12)
    plt.title('완치 예측에 중요한 특성', fontsize=14)
    
    # 양수/음수 구분을 위한 범례
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='g', label='완치와 양의 상관관계'),
        Patch(facecolor='r', label='완치와 음의 상관관계')
    ]
    plt.legend(handles=legend_elements)
    
    plt.tight_layout()
    plt.show()
    
    # 중요도 출력
    print("특성 중요도 (상위):")
    for feature, importance in sorted_importance[:10]:
        sign = "+" if importance > 0 else "-"
        print(f"  {sign} {feature}: {abs(importance):.4f}")
    
    return plt

### 신규 환자 적용

#### 신규 환자 훼복 궤적 예측

In [18]:
def predict_recovery_trajectory(patient_data, df_with_probs, model, avg_stage_patterns, cluster_centers):
    """신규 환자의 회복 궤적 예측"""
    import numpy as np
    from scipy.spatial.distance import cosine
    from scipy.special import softmax
    
    # 환자 ID 확인
    patient_id = patient_data['환자번호']
    
    # 환자 방문 데이터 추출
    patient_visits = df_with_probs[df_with_probs['환자번호'] == patient_id].sort_values('날짜')
    
    if len(patient_visits) == 0:
        raise ValueError(f"환자 ID {patient_id}에 대한 데이터를 찾을 수 없습니다.")
    
    # 가장 최근 방문 사용
    latest_visit = patient_visits.iloc[-1]
    
    # 군집 확률 추출
    n_clusters = len(cluster_centers)
    cluster_probs = {
        f'cluster_{i}_prob': latest_visit[f'cluster_{i}_prob'] 
        for i in range(n_clusters) if f'cluster_{i}_prob' in latest_visit
    }
    
    # 특성 및 타겟 데이터 준비
    feature_cols = list(model.feature_names_in_)
    
    # 특성 벡터 구성
    features = []
    for col in feature_cols:
        if col in latest_visit:
            features.append(latest_visit[col])
        else:
            features.append(0)  # 누락된 특성은 0으로 대체
    
    features_array = np.array([features])
    
    # 회복 가능성 예측
    recovery_prob = model.predict_proba(features_array)[0][1]
    
    # 현재 군집 확률 분포 추출
    current_probs = np.array([[latest_visit[f'cluster_{i}_prob'] for i in range(n_clusters)]])
    
    # 가장 유사한 진행 단계 찾기
    min_distance = float('inf')
    best_stage = None
    
    for stage, avg_probs in avg_stage_patterns.items():
        stage_probs = np.array([[avg_probs.get(f'cluster_{i}_prob', 0) for i in range(n_clusters)]])
        
        # 유클리드 거리 계산
        distance = np.linalg.norm(current_probs - stage_probs)
        
        if distance < min_distance:
            min_distance = distance
            best_stage = stage
    
    # 주요 임상 지표 현재 값
    clinical_metrics = {}
    for metric in ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']:
        if metric in latest_visit:
            clinical_metrics[metric] = float(latest_visit[metric])
    
    # 완치 군집 추세 대비 현재 위치 시각화
    visualize_patient_vs_recovery_trend(current_probs[0], avg_stage_patterns, n_clusters, best_stage)
    
    # 결과 반환
    prediction_result = {
        'patient_id': patient_id,
        'recovery_probability': float(recovery_prob),
        'current_stage': best_stage,
        'cluster_probabilities': cluster_probs,
        'most_likely_cluster': int(np.argmax([cluster_probs.get(f'cluster_{i}_prob', 0) for i in range(n_clusters)])),
        'estimated_remaining_stages': 10 - (best_stage // 10) if best_stage is not None else None,
        'current_clinical_metrics': clinical_metrics
    }
    
    return prediction_result

In [19]:
def visualize_patient_vs_recovery_trend(current_probs, avg_stage_patterns, n_clusters, current_stage):
    """환자의 현재 군집 확률과 완치 환자의 트렌드 비교 시각화"""
    import matplotlib.pyplot as plt
    
    # 데이터 준비
    stages = sorted(avg_stage_patterns.keys())
    recovery_probs = {}
    
    for i in range(n_clusters):
        col = f'cluster_{i}_prob'
        recovery_probs[col] = [avg_stage_patterns[stage].get(col, 0) for stage in stages]
    
    # 시각화
    plt.figure(figsize=(12, 6))
    
    # 전체 트렌드 그리기
    for i in range(n_clusters):
        col = f'cluster_{i}_prob'
        plt.plot(stages, recovery_probs[col], '--', alpha=0.7, linewidth=1, label=f'군집 {i} 트렌드')
    
    # 현재 위치 강조
    current_x = current_stage if current_stage is not None else 0
    for i in range(n_clusters):
        col = f'cluster_{i}_prob'
        plt.scatter([current_x], [current_probs[i]], s=100, label=f'현재 군집 {i}')
    
    # 예상 경로 표시
    if current_stage is not None:
        future_stages = [s for s in stages if s > current_stage]
        for i in range(n_clusters):
            col = f'cluster_{i}_prob'
            future_probs = [avg_stage_patterns[stage].get(col, 0) for stage in future_stages]
            if future_stages and future_probs:
                plt.plot(future_stages, future_probs, 'g-', alpha=0.8, linewidth=2)
        
        # 현재 위치 강조
        plt.axvline(x=current_stage, color='r', linestyle=':', alpha=0.6, label='현재 단계')
    
    plt.title('환자의 현재 군집 확률 vs 완치 환자 트렌드', fontsize=14)
    plt.xlabel('치료 진행 단계 (%)', fontsize=12)
    plt.ylabel('군집 소속 확률', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
    plt.xticks(stages)
    plt.tight_layout()
    plt.show()
    
    return plt

### 성능 평가

In [20]:
def evaluate_model_statistics(df_with_probs, model, recovered_ids):
    """모델의 통계적 성능 평가"""
    from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
    import matplotlib.pyplot as plt
    import numpy as np
    
    # 실제 완치 상태
    df_with_probs['is_recovered'] = df_with_probs['환자번호'].isin(recovered_ids)
    
    # 특성 컬럼
    feature_cols = list(model.feature_names_in_)
    
    # 예측 확률
    X = df_with_probs[feature_cols].fillna(0)
    y_true = df_with_probs['is_recovered']
    y_pred_proba = model.predict_proba(X)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # ROC 곡선
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 1, 1)
    plt.plot(fpr, tpr, label=f'ROC 곡선 (AUC = {roc_auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('완치 예측 모델 ROC 곡선')
    plt.legend(loc='lower right')
    
    # 혼동 행렬
    cm = confusion_matrix(y_true, y_pred)
    plt.subplot(2, 1, 2)
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('혼동 행렬')
    plt.colorbar()
    
    classes = ['비완치', '완치']
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes)
    plt.yticks(tick_marks, classes)
    
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.ylabel('실제 레이블')
    plt.xlabel('예측 레이블')
    plt.tight_layout()
    plt.show()
    
    # 분류 보고서
    report = classification_report(y_true, y_pred, target_names=classes)
    print("분류 보고서:")
    print(report)
    
    # 통계적 유의성 검정
    from scipy import stats
    
    # 완치 환자와 비완치 환자의 예측 확률 비교
    recovered_probs = y_pred_proba[y_true]
    non_recovered_probs = y_pred_proba[~y_true]
    
    # t-검정
    t_stat, p_value = stats.ttest_ind(recovered_probs, non_recovered_probs)
    print(f"\n완치/비완치 환자 예측 확률 t-검정:")
    print(f"t-통계량: {t_stat:.4f}, p-값: {p_value:.4f}")
    print(f"통계적 유의성: {'있음 (p < 0.05)' if p_value < 0.05 else '없음 (p >= 0.05)'}")
    
    return {
        'roc_auc': roc_auc,
        'confusion_matrix': cm,
        'classification_report': report,
        't_test': {'t_stat': t_stat, 'p_value': p_value}
    }

### 실행 파이프라인

In [21]:
def run_complete_predictive_pipeline(df, api_key, n_clusters=5, temperature=0.1, recovery_threshold=70):
    """전체 회복 궤적 예측 파이프라인 실행 함수
    
    Parameters:
    -----------
    df : DataFrame
        환자 데이터가 포함된 데이터프레임
    api_key : str
        OpenAI API 키
    n_clusters : int, optional (default=5)
        생성할 군집 수
    temperature : float, optional (default=0.1)
        확률 계산 시 사용할 온도 매개변수 (낮을수록 더 극단적 확률 분포)
    recovery_threshold : float, optional (default=70)
        완치로 간주할 복합 점수 임계값
        
    Returns:
    --------
    dict
        모든 모델 구성요소 및 예측 함수를 포함하는 결과 사전
    """
    print("===== 턱관절 장애 환자 회복 궤적 예측 시스템 구축 =====")
    print(f"군집 수: {n_clusters}, 온도 매개변수: {temperature}, 완치 임계값: {recovery_threshold}")
    
    # 1. 방문별 임베딩 및 군집화
    print("\n[단계 1/5] 방문별 임베딩 생성 및 군집화...")
    df_with_probs, cluster_centers, cluster_features = run_visit_level_clustering_pipeline(
        df, api_key, n_clusters=n_clusters, temperature=temperature
    )
    
    # 2. 완치 환자 식별
    print("\n[단계 2/5] 완치 환자 식별...")
    recovered_ids, patient_details = identify_recovered_patients(df, recovery_threshold)
    
    # 3. 완치 환자 궤적 분석
    print("\n[단계 3/5] 완치 환자 궤적 분석...")
    if len(recovered_ids) > 0:
        avg_stage_patterns = analyze_recovered_patient_trajectories(
            df_with_probs, recovered_ids, n_clusters
        )
    else:
        print("경고: 식별된 완치 환자가 없습니다. 샘플 궤적을 사용합니다.")
        # 샘플 궤적 생성 (실제 데이터 없는 경우)
        avg_stage_patterns = create_sample_trajectories(n_clusters)
    
    # 4. 예측 모델 구축
    print("\n[단계 4/5] 회복 예측 모델 구축...")
    model, feature_importance = build_recovery_prediction_model(
        df_with_probs, recovered_ids, ['CC_vas', 'CMO_before', 'MMO_before', 'dif_MMO_CMO']
    )
    
    # 5. 모델 평가 및 통계적 검증
    print("\n[단계 5/5] 모델 통계적 검증...")
    model_stats = evaluate_model_statistics(df_with_probs, model, recovered_ids)
    
    # 6. 예측 시스템 구축 완료
    print("\n===== 예측 시스템 구축 완료 =====")
    
    # 예측 함수 래핑
    def predict_recovery(patient_id):
        patient_data = df[df['환자번호'] == patient_id].iloc[-1]
        return predict_recovery_trajectory(
            patient_data, df_with_probs, model, avg_stage_patterns, cluster_centers
        )
    
    # 결과 반환
    result = {
        'df_with_probs': df_with_probs,
        'cluster_centers': cluster_centers,
        'cluster_features': cluster_features,
        'recovered_ids': recovered_ids,
        'patient_details': patient_details,
        'avg_stage_patterns': avg_stage_patterns,
        'model': model,
        'feature_importance': feature_importance,
        'model_stats': model_stats,
        'predict_recovery': predict_recovery
    }
    
    return result

In [ ]:
def run_example_prediction(prediction_system, patient_id):
    """예제 환자에 대한 예측 실행 및 결과 출력"""
    print(f"\n===== 환자 {patient_id}의 회복 궤적 예측 =====")
    
    try:
        # 예측 수행
        result = prediction_system['predict_recovery'](patient_id)
        
        # 결과 출력
        print(f"환자 ID: {result['patient_id']}")
        print(f"완치 확률: {result['recovery_probability']:.2%}")
        print(f"현재 단계: {result['current_stage']}% (완치 기준)")
        
        if 'estimated_remaining_stages' in result and result['estimated_remaining_stages'] is not None:
            print(f"예상 남은 단계: {result['estimated_remaining_stages']} (각 단계는 약 10%의 진행 의미)")
        
        print("\n현재 군집 소속 확률:")
        for cluster, prob in result['cluster_probabilities'].items():
            print(f"  - {cluster}: {prob:.2%}")
        
        print(f"\n가장 유사한 군집: {result['most_likely_cluster']}")
        
        print("\n현재 임상 지표:")
        for metric, value in result['current_clinical_metrics'].items():
            print(f"  - {metric}: {value}")
        
        # 추가 통계 정보가 있으면 출력
        if 'statistical_validity' in result:
            print("\n통계적 유의성:")
            print(f"  - p값: {result['statistical_validity']['p_value']:.4f}")
            print(f"  - 유의성: {'있음 (p<0.05)' if result['statistical_validity']['p_value'] < 0.05 else '없음 (p≥0.05)'}")
    
    except Exception as e:
        print(f"예측 오류: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
def create_interactive_dashboard(prediction_system, df):
    """예측 시스템의 결과를 보여주는 대화형 대시보드 생성 (Optional)"""
    try:
        import matplotlib.pyplot as plt
        from ipywidgets import interact, widgets
        import ipywidgets as widgets
        
        # 환자 목록 준비
        patient_ids = sorted(df['환자번호'].unique())
        
        # 환자 선택 위젯
        patient_dropdown = widgets.Dropdown(
            options=patient_ids,
            description='환자 선택:',
            style={'description_width': 'initial'}
        )
        
        # 예측 실행 함수
        def on_patient_change(patient_id):
            if patient_id:
                plt.close('all')  # 이전 그래프 닫기
                run_example_prediction(prediction_system, patient_id)
        
        # 위젯 연결
        interact(on_patient_change, patient_id=patient_dropdown)
        
        print("대화형 대시보드가 준비되었습니다. 환자를 선택하여 예측 결과를 확인하세요.")
    
    except ImportError:
        print("대화형 대시보드를 위해 ipywidgets가 필요합니다.")
        print("pip install ipywidgets를 실행하여 설치할 수 있습니다.")
        
        # 대체 기능: 몇 가지 예제 환자 예측
        sample_patients = df['환자번호'].unique()[:3]  # 처음 3명만
        for patient_id in sample_patients:
            run_example_prediction(prediction_system, patient_id)

### 샘플 실험

In [64]:
df.환자번호.nunique()
df_sp = df.groupby('환자번호').size().reset_index(name='방문수')
df_sp['방문수_범주'] = df_sp['방문수'].apply(lambda x: '낮음' if x <= 3 else ('중간' if x <= 10 else '높음'))
df_sp.groupby('방문수_범주').size()
# 방문수_범주 분포를 반영한 환자번호 샘플링 (계층적 샘플링)

# 각 범주별 환자 수 확인
category_counts = df_sp.groupby('방문수_범주').size()

# 계층적 샘플링(stratified sampling) 수행
from sklearn.model_selection import StratifiedShuffleSplit

# 전체 샘플 수 설정
total_samples = 600  # 총 샘플링할 환자 수

# 계층적 샘플링 수행
stratified_split = StratifiedShuffleSplit(n_splits=1, test_size=total_samples/len(df_sp), random_state=42)
for _, sample_idx in stratified_split.split(df_sp, df_sp['방문수_범주']):
    stratified_sample = df_sp.iloc[sample_idx]

stratified_sample.환자번호.to_list()
df_sp = df[df['환자번호'].isin(stratified_sample.환자번호.to_list())]


In [ ]:
# def main():
#     """전체 파이프라인 실행 및 테스트"""
#     import os
#     import pandas as pd
#     from datetime import datetime
    
#     # 1. 데이터 로드
#     print("데이터 로드 중...")
#     df = pd.read_hdf('/Users/nam-yeong/git/prj_centum/gpt_word/final_result/preprocessed_final_df.h5', key='df')
#     print(f"로드된 데이터 크기: {df.shape}")
    
#     # 2. API 키 설정
#     api_key = os.environ.get('OPENAI_API_KEY')
#     if not api_key:
#         api_key = input("OpenAI API 키를 입력하세요: ")
    
#     # 3. 전체 파이프라인 실행
#     prediction_system = run_complete_predictive_pipeline(
#         df, api_key, n_clusters=5, temperature=0.1, recovery_threshold=70
#     )
    
#     # 4. 예측 시스템 테스트
#     # 4.1 임의의 환자 선택
#     sample_patient = df['환자번호'].sample(1).iloc[0]
#     run_example_prediction(prediction_system, sample_patient)
    
#     # 4.2 (선택적) 대화형 대시보드 생성
#     create_interactive_dashboard(prediction_system, df)
    
#     # 5. 결과 저장
#     timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
#     result_path = f'/Users/nam-yeong/git/prj_centum/gpt_word/final_result/prediction_model_{timestamp}.pkl'
    
#     import pickle
#     with open(result_path, 'wb') as f:
#         # 저장이 불가능한 콜백 함수 등 제외
#         save_dict = {k: v for k, v in prediction_system.items() 
#                     if k not in ['predict_recovery']}
#         pickle.dump(save_dict, f)
    
#     print(f"\n예측 모델이 '{result_path}'에 저장되었습니다.")
#     print("\n실행 완료!")

# if __name__ == "__main__":
    # embedding_batch_size = 100
    
    # main(embedding_batch_size)